## Учёт заказов в кофейне

В небольшой кофейне сотрудники принимают заказы от клиентов.
Нужно сделать консольную программу для учёта заказов, их статусов и оплаты.

### Что должна уметь программа

Меню:
```txt
1. Добавить заказ
2. Показать все заказы
3. Изменить статус заказа
4. Показать активные заказы
5. Найти заказы по имени клиента
6. Отметить заказ как оплаченный
7. Показать неоплаченные заказы
8. Показать общую выручку
0. Выход
```

### Данные одного заказа
```txt
Номер заказа
Имя клиента
Состав заказа
Стоимость
Статус
Оплачен / не оплачен
Приоритет
```

### Статусы заказа
```txt
новый
готовится
готов
выдан
отменён
```

### Приоритет заказа
```txt
обычный
срочный
```

### Пример заказа
```txt
№1
Клиент: Анна
Заказ: Капучино и круассан
Стоимость: 450 руб.
Статус: новый
Оплата: не оплачен
Приоритет: обычный
```

### Обязательные условия

1. Номер заказа создаётся автоматически.
2. При добавлении заказа статус всегда новый.
3. При добавлении заказ всегда считается не оплачен.
4. Стоимость заказа должна быть больше 0.
5. Можно изменить статус только на:
   - новый
   - готовится
   - готов
   - выдан
   - отменён
6. Заказ нельзя сделать выданным, если он не оплачен.
7. Отменённый заказ нельзя изменить на другой статус.
8. Активные заказы — это заказы со статусом новый, готовится или готов.
9. Общая выручка считается только по оплаченным заказам, которые не были отменены.
10. При поиске по имени клиента программа должна показывать все заказы этого клиента.
11. Срочные заказы при выводе активных заказов должны показываться первыми.


### Требования к ООП и SOLID

Решение должно быть реализовано в объектно-ориентированном стиле с соблюдением принципов SOLID:

1. Отдельный класс для заказа.
2. Отдельный сервис для управления заказами.
3. Отдельный класс для работы с консольным меню.
4. Отдельная логика проверки статусов и оплаты.
5. Классы не должны брать на себя лишние обязанности.
6. Зависимости между частями программы должны быть минимальными и понятными.

In [ ]:
from dataclasses import dataclass
from time import time
statuses = {'new', 'in_proccess', 'ready', 'issued', 'canceled'}

@dataclass
class Order:
    id: str
    client: str
    content: str
    cost: int
    priority: str
    status: str = 'new'
    is_payed: bool = False
    
    def change_status(self, status):
        if status not in statuses:
            print('Неккоректный статус!')
            return
        if status == 'issued' and self.is_payed is False:
            print('Заказ должен быть оплачен!')
            return
        if self.status == 'canceled':
            print('Отменненый заказ нельзя изменять!')
        self.status = status
    
    def mark_payed(self):
        self.is_payed = True



class OrderService:
    def __init__(self):
        self.orders = {}
    def add_order(self, client, content, cost, priority):
        order_id = "ORD-"+str(time()).split('.')[1][:2]
        if cost <= 0:
            print('Неккоректная стоимость!')
            return
        if priority not in {'basic', 'urgent'}:
            print('Неккоректный приоритет!')
            return
        self.orders[order_id] = Order(order_id, client, content, cost, priority=priority)
    
    def show_orders(self):
        print('Все заказы: ')
        for order in self.orders.values():
            print(f'{order.id} | {order.client} | {order.content} | {order.cost} | {order.status} | {order.is_payed} | {order.priority}')

    def change_status(self, order_id, status):
        order = self.orders[order_id]
        order.change_status(status)
    
    def show_active_orders(self):
        print('Активные заказы: ')
        arr_urgent = []
        arr_basic = []
        for order in self.orders.values():
            if order.status in {'new', 'in_proccess', 'ready'}:
                if order.priority == 'urgent':
                    arr_urgent.append(order)
                    continue
                arr_basic.append(order)
        arr = arr_urgent + arr_basic
        for order in arr:
            print(f'{order.id} | {order.client} | {order.content} | {order.cost} | {order.status} | {order.is_payed} | {order.priority}')

                
    def find_by_client(self, client):
        print(f'Заказы с именем {client}')
        for order in self.orders.values():
            if order.client == client:
                print(f'{order.id} | {order.client} | {order.content} | {order.cost} | {order.status} | {order.is_payed} | {order.priority}')

    def mark_as_payed(self, order_id):
        order = self.orders[order_id]
        order.mark_payed()

    def show_not_payed(self):
        print('Все неоплаченные заказы: ')
        for order in self.orders.values():
            if order.is_payed == False:
                print(f'{order.id} | {order.client} | {order.content} | {order.cost} | {order.status} | {order.is_payed} | {order.priority}')
    
    def show_revenue(self):
        res = 0
        for order in self.orders.values():
            if order.is_payed == True and order.status != 'canceled':
                res += order.cost
        print(f'Общая выручка: {res}')
    

service = OrderService()
class Menu:
    @staticmethod
    def create_order():
        client  = input('Введите клиента: ')
        content = input('Введите содержимое заказа: ')
        cost = int(input('Введите стоимость: '))
        priority = input('Введите приоритет: ')
        service.add_order(client, content, cost, priority)

    def change_status():
        order_id = input('Введите order_id: ')
        status = input('Введите новый статус: ')

        service.change_status(order_id, status)
    
    def find():
        name = input('Введите имя клиента: ')
        service.find_by_client(name)
    
    def mark():
        order_id = input('Введите order_id: ')
        service.mark_as_payed(order_id)
    def main():
        while True:
            print("""
1.Добавить заказ
2.Показать все заказы
3.Изменить статус заказа
4.Показать активные заказы
5.Найти заказы по имени клиента
6.Отметить заказ как оплаченный
7.Показать неоплаченные заказы
8.Показать общую выручку
0.выход
""")
            ch = input('Выберите действие: ')
            if ch == '1':
                
                Menu.create_order()
            if ch == '2':
                service.show_orders()
            if ch == '3':
                Menu.change_status()
            if ch == '4':
                service.show_active_orders()
            if ch == '5':
                Menu.find()
            if ch == '6':
                Menu.mark()
            if ch == '7':
                service.show_not_payed()
            if ch == '8':
                service.show_revenue()
            if ch == '0':
                break
            





Menu.main()




1.Добавить заказ
2.Показать все заказы
3.Изменить статус заказа
4.Показать активные заказы
5.Найти заказы по имени клиента
6.Отметить заказ как оплаченный
7.Показать неоплаченные заказы
8.Показать общую выручку
0.выход



Выберите действие:  1
Введите клиента:  cl1
Введите содержимое заказа:  coffee
Введите стоимость:  100
Введите приоритет:  basic



1.Добавить заказ
2.Показать все заказы
3.Изменить статус заказа
4.Показать активные заказы
5.Найти заказы по имени клиента
6.Отметить заказ как оплаченный
7.Показать неоплаченные заказы
8.Показать общую выручку
0.выход



Выберите действие:  2


Все заказы: 
ORD-14 | cl1 | coffee | 100 | new | False | basic

1.Добавить заказ
2.Показать все заказы
3.Изменить статус заказа
4.Показать активные заказы
5.Найти заказы по имени клиента
6.Отметить заказ как оплаченный
7.Показать неоплаченные заказы
8.Показать общую выручку
0.выход



...

Общая выручка: 100

1.Добавить заказ
2.Показать все заказы
3.Изменить статус заказа
4.Показать активные заказы
5.Найти заказы по имени клиента
6.Отметить заказ как оплаченный
7.Показать неоплаченные заказы
8.Показать общую выручку
0.выход



Выберите действие:  3
Введите order_id:  ORD-14
Введите новый статус:  in_proccess



1.Добавить заказ
2.Показать все заказы
3.Изменить статус заказа
4.Показать активные заказы
5.Найти заказы по имени клиента
6.Отметить заказ как оплаченный
7.Показать неоплаченные заказы
8.Показать общую выручку
0.выход



Выберите действие:  2


Все заказы: 
ORD-14 | cl1 | coffee | 100 | in_proccess | True | basic
ORD-29 | cl2 | tea | 300 | new | False | urgent

1.Добавить заказ
2.Показать все заказы
3.Изменить статус заказа
4.Показать активные заказы
5.Найти заказы по имени клиента
6.Отметить заказ как оплаченный
7.Показать неоплаченные заказы
8.Показать общую выручку
0.выход

